# Stage 05 — Assemble and smoke-test the Ecuador 2022 network

This stage attaches the accepted Stage 1 demand and Phase 4 generator/profile products to the supported transmission topology. It records every spatial aggregation decision, preserves annual demand and capacity exactly, adds explicitly labeled emergency load-shedding generators, and serializes an **unsolved candidate** network. No optimization is run here.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import matplotlib.pyplot as plt
import pandas as pd
import pypsa

def find_repository_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'src').is_dir() and (candidate / 'config/artifacts.yaml').is_file():
            return candidate
    raise RuntimeError('Could not locate the EcuadorElectricGrid repository root.')

ROOT = find_repository_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.assembly import (
    GENERATOR_ATTACHMENT_BANDS, LOAD_ATTACHMENT_BANDS, assemble_2022_network,
    assert_reload_equivalent, evaluate_attachment_materiality, read_hourly_table,
    summarize_attachment_screen,
)
from src.artifacts import format_preflight_report, run_preflight
from src.topology import build_2022_topology, load_repair_table

preflight = run_preflight('05_assembly', root=ROOT, offline=True)
print(format_preflight_report(preflight))
preflight.raise_for_errors()


## Load reviewed inputs

Demand is already in MW and sums to the official 27,561.62 GWh control. The generator fleet contains 133 existing SNI rows. Hydro and variable-renewable tables contain the complete 8,760-hour Phase 4 availability assumptions. Technology costs are modeling assumptions used only to prepare Phase 6 dispatch.

In [ ]:
paths = {
    'base': ROOT / 'data/raw/networks/base.nc',
    'repairs': ROOT / 'data/interim/network/topology_repairs_2022.csv',
    'demand': ROOT / 'data/processed/demand/demand_profiles_2022_mw.csv',
    'generators': ROOT / 'data/processed/generation/generators_2022.csv',
    'location_overrides': ROOT / 'data/raw/generation/plant_location_overrides_2022.csv',
    'hydro': ROOT / 'data/processed/generation/hydro_availability_2022.csv',
    'renewable': ROOT / 'data/processed/generation/renewable_availability_2022.csv',
    'costs': ROOT / 'data/raw/generation/technology_cost.csv',
    'carriers': ROOT / 'data/raw/generation/carriers.csv',
    'generator_connections': ROOT / 'data/interim/network/generator_connection_evidence_2022.csv',
    'load_connections': ROOT / 'data/interim/network/load_connection_evidence_2022.csv',
}
base = pypsa.Network(paths['base'])
topology = build_2022_topology(base, load_repair_table(paths['repairs'])).network
demand = read_hourly_table(paths['demand'])
generators = pd.read_csv(paths['generators'])
location_overrides = pd.read_csv(paths['location_overrides'])
hydro = read_hourly_table(paths['hydro'])
renewable = read_hourly_table(paths['renewable'])
costs = pd.read_csv(paths['costs'], comment='#')
carriers = pd.read_csv(paths['carriers'], index_col=0)
generator_connection_evidence = pd.read_csv(paths['generator_connections'])
load_connection_evidence = pd.read_csv(paths['load_connections'])


## Explicit spatial attachments

Existing 138/230/500 kV bus IDs are retained exactly. This notebook first reconstructs the original geographic-nearest-bus assignment, including the superseded Termoesmeraldas coordinates, so the problem remains auditable. It then applies the reviewed generator and load connection tables. Termoesmeraldas I and II move to the verified 138 kV side of S/E Esmeraldas (bus 42). Eleven flagged 69 kV loads use a modeled electrical path or official utility-route interpretation; load 262 uses the owner's requested nearest-138-kV fallback (bus 52, about 56.65 km) and remains unresolved.

Distance alone is not a useful severity measure: a remote 1 MW plant is less material than a remote 1,500 MW plant. The diagnostic therefore screens generator distance by installed MW and load distance by annual GWh. These unchanged thresholds are Phase 5 review conventions, not physical interconnection standards. A long distance can be explained by an explicitly reviewed electrical aggregation, but its raw distance flag is retained. Load 262 is not accepted by either test and remains `not acceptable - fix this bus`.

The row-level evidence table cites the exact source locator. Primary utility evidence includes the [EMELNORTE 2022-2025 plan](https://www.emelnorte.com/docs/lotaip/planificacion/PLAN_ESTRATEGICO_EMELNORTE_2022-2025-signed-signed.pdf), [CNEL Palenque/Vinces geospatial layer](https://geoportal.cnelep.gob.ec/arcgis/rest/services/REP/CNELEPN/FeatureServer/17), [CNEL Los Ríos record](https://www.cnelep.gob.ec/mas-de-60-profesionales-electricos-realizaron-mantenimiento-preventivo-en-los-rios/), [CNEL Naranjal terms of reference](https://www.cnelep.gob.ec/wp-content/uploads/2016/04/Termino-de-Referencia.pdf), and [CNEL San Lorenzo record](https://www.cnelep.gob.ec/wp-content/uploads/2020/05/literal_i-proceso_de_contrataciones-4.pdf). Graph-derived paths cite the checksum-pinned upstream `base.nc` and remain modeling evidence rather than official facts.

In [ ]:
original_generators = generators.copy()
for override in location_overrides.itertuples(index=False):
    mask = original_generators['name'].eq(override.plant_original)
    original_generators.loc[mask, ['latitude', 'longitude']] = [override.superseded_latitude, override.superseded_longitude]
original_result = assemble_2022_network(
    topology, base.buses, demand, original_generators, hydro, renewable, costs, carriers,
)
original_load_screen, original_generator_screen = evaluate_attachment_materiality(
    original_result.load_attachments, original_result.generator_attachments, demand, original_generators
)
result = assemble_2022_network(
    topology, base.buses, demand, generators, hydro, renewable, costs, carriers,
    generator_connection_evidence, load_connection_evidence,
    provenance_paths={key: value for key, value in paths.items() if key not in {'base', 'carriers'}},
)
load_screen, generator_screen = evaluate_attachment_materiality(
    result.load_attachments, result.generator_attachments, demand, generators
)
load_screen['reviewed_assignment_status'] = load_screen['connection_review_status']
load_summary = summarize_attachment_screen(load_screen, 'consumption_band', 'annual_consumption_gwh')
generator_summary = summarize_attachment_screen(generator_screen, 'capacity_band', 'p_nom_mw')
display(pd.DataFrame(GENERATOR_ATTACHMENT_BANDS, columns=['upper_MW', 'capacity_band', 'acceptable_km']))
display(pd.DataFrame(LOAD_ATTACHMENT_BANDS, columns=['upper_GWh_per_year', 'consumption_band', 'acceptable_km']))
display(pd.DataFrame({'diagnostic': ['original load distance flags', 'corrected load distance flags', 'corrected loads still requiring source validation', 'original generator distance flags', 'corrected generator distance flags'], 'count': [(original_load_screen.screening_status != 'acceptable').sum(), (load_screen.screening_status != 'acceptable').sum(), (load_screen.connection_review_status == 'not_acceptable_requires_source_validation').sum(), (original_generator_screen.screening_status != 'acceptable').sum(), (generator_screen.screening_status != 'acceptable').sum()]}))
display(generator_summary)
display(load_summary)
display(generator_screen.query("screening_status != 'acceptable'").sort_values('distance_capacity_index_mw_km', ascending=False))
display(load_screen.query("screening_status != 'acceptable'").sort_values('distance_energy_index_gwh_km', ascending=False))
display(load_screen.query("connection_review_status == 'not_acceptable_requires_source_validation'"))
result.fingerprint


## Save audits and validation figures

The before/after plots show distance against the materiality that makes a questionable attachment consequential. Red crosses fail the unchanged distance limit; blue circles pass. In the corrected load plot, green rings identify long-distance rows whose electrical aggregation has been reviewed, while load 262 remains a red unresolved point. Original and corrected CSVs preserve both states. Emergency generators are excluded from real capacity and will be reported separately as load shedding in Phase 6.

In [ ]:
audit_dir = ROOT / 'data/interim/network'
figure_dir = ROOT / 'reports/figures/phase_05'
network_dir = ROOT / 'data/processed/networks'
for directory in (audit_dir, figure_dir, network_dir):
    directory.mkdir(parents=True, exist_ok=True)
result.load_attachments.to_csv(audit_dir / 'load_attachments_2022.csv', index=False)
result.generator_attachments.to_csv(audit_dir / 'generator_attachments_2022.csv', index=False)
original_load_screen.to_csv(audit_dir / 'load_attachment_screen_original_2022.csv', index=False)
original_generator_screen.to_csv(audit_dir / 'generator_attachment_screen_original_2022.csv', index=False)
load_screen.to_csv(audit_dir / 'load_attachment_screen_2022.csv', index=False)
generator_screen.to_csv(audit_dir / 'generator_attachment_screen_2022.csv', index=False)
with (audit_dir / 'assembled_network_fingerprint_2022.json').open('w', encoding='utf-8') as handle:
    json.dump(result.fingerprint, handle, indent=2, sort_keys=True)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
result.load_attachments.distance_km.plot.hist(ax=axes[0], bins=25, color='#4c78a8')
result.generator_attachments.distance_km.plot.hist(ax=axes[1], bins=25, color='#f58518')
axes[0].set_title('Load-to-transmission attachment distance')
axes[1].set_title('Generator-to-transmission attachment distance')
for ax in axes:
    ax.set_xlabel('Great-circle distance [km]')
    ax.set_ylabel('Count')
fig.tight_layout()
fig.savefig(figure_dir / 'attachment_distances_2022.png', dpi=180, bbox_inches='tight')
plt.show()

def plot_materiality_pair(generator_data, load_data, title, destination, show_review=False):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, screen, x, xlabel in [(axes[0], generator_data, 'p_nom_mw', 'Generator nominal capacity [MW]'), (axes[1], load_data, 'annual_consumption_gwh', 'Load annual consumption [GWh/year]')]:
        accepted = screen.screening_status.eq('acceptable')
        ax.scatter(screen.loc[accepted, x], screen.loc[accepted, 'distance_km'], s=24, alpha=0.65, label='within distance screen', color='#377eb8')
        ax.scatter(screen.loc[~accepted, x], screen.loc[~accepted, 'distance_km'], s=55, marker='x', linewidth=1.8, label='distance flag', color='#d62728')
        if show_review and x == 'annual_consumption_gwh':
            reviewed = screen.connection_review_status.eq('acceptable_electrical_aggregation') & ~accepted
            ax.scatter(screen.loc[reviewed, x], screen.loc[reviewed, 'distance_km'], s=105, facecolors='none', edgecolors='#2ca02c', linewidth=1.8, label='reviewed electrical aggregation')
            unresolved = screen.connection_review_status.eq('not_acceptable_requires_source_validation')
            for row in screen.loc[unresolved].itertuples():
                ax.annotate(row.load_id, (getattr(row, x), row.distance_km), xytext=(5, 5), textcoords='offset points', fontsize=8)
        ordered = screen.sort_values(x)
        ax.step(ordered[x], ordered.acceptable_distance_km, where='post', color='#222222', linestyle='--', label='screening limit')
        ax.set_xscale('log'); ax.set_xlabel(xlabel); ax.set_ylabel('Attachment distance [km]'); ax.grid(alpha=0.2); ax.legend(fontsize=7)
    axes[0].set_title('Generator assignments'); axes[1].set_title('Load assignments')
    fig.suptitle(title); fig.tight_layout(); fig.savefig(destination, dpi=180, bbox_inches='tight'); plt.show()
plot_materiality_pair(original_generator_screen, original_load_screen, 'Original nearest-bus diagnosis', figure_dir / 'attachment_materiality_screen_original_2022.png')
plot_materiality_pair(generator_screen, load_screen, 'Corrected assignments and remaining review', figure_dir / 'attachment_materiality_screen_corrected_2022.png', show_review=True)


## Serialization smoke test and handoff

The candidate must reload with identical component IDs, structure, demand, and availability. This is a serialization smoke test only. The annual optimization, energy balance, carrier comparison, calibration log, and 5% acceptance gate remain Phase 6 work.

In [ ]:
candidate_path = network_dir / 'network_base_filled.nc'
result.network.export_to_netcdf(candidate_path)
reloaded = pypsa.Network(candidate_path)
assert_reload_equivalent(result.network, reloaded)
reloaded.consistency_check()
artifact_sha256 = hashlib.sha256(candidate_path.read_bytes()).hexdigest()
artifact_record = {'path': str(candidate_path.relative_to(ROOT)), 'bytes': candidate_path.stat().st_size, 'sha256': artifact_sha256, 'status': 'unsolved Phase 5 candidate'}
with (audit_dir / 'assembled_network_artifact_2022.json').open('w', encoding='utf-8') as handle:
    json.dump(artifact_record, handle, indent=2, sort_keys=True)
print(artifact_record)
